In [12]:
import arcpy

In [ ]:
# ── PATHS ──────────────────────────────────────────────────────────────────────
csv_in     = r"C:\Users\kkeii\OneDrive\Documents\THESIS_REPOSITORIES\MISSION_REPOSITORIES\UVI2_Around_STX_20250306\data\4_data_interpolated\LF_background_INTERPOLATED_UVI2_Around_STX_20250306.csv"

gdb        = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Projects\THESIS_PROJECTS\map_setup_UVI2_Around_STX_20250306\map_setup_UVI2_Around_STX_20250306.gdb"
aprx_path  = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Projects\THESIS_PROJECTS\map_setup_UVI2_Around_STX_20250306\map_setup_UVI2_Around_STX_20250306.aprx"

pts_wgs84 = gdb + r"\LF_deploy_point_WGS84"
pts_nad83 = gdb + r"\LF_deploy_point_NAD83"

geog_transform = "WGS_1984_(ITRF00)_To_NAD_1983"

aprx       = arcpy.mp.ArcGISProject("CURRENT")
active_map = aprx.activeMap

In [ ]:
# ── 1. ADD CSV TO MAP ──────────────────────────────────────────────────────────
csv_layer = active_map.addDataFromPath(csv_in)
print(f"Added to map: {csv_layer.name}")

In [ ]:
# ── 2. XY TABLE TO POINT ───────────────────────────────────────────────────────
arcpy.management.XYTableToPoint(
    in_table          = csv_layer,
    out_feature_class = pts_wgs84,
    x_field           = "start_longitude_DD",
    y_field           = "start_latitude_DD",
    z_field           = None,
    coordinate_system = arcpy.SpatialReference(4326)
)
print(f"Created: {pts_wgs84}")

In [ ]:
# ── 3. PROJECT POINTS ──────────────────────────────────────────────────────────
arcpy.management.Project(
    in_dataset              = pts_wgs84,
    out_dataset             = pts_nad83,
    out_coor_system         = arcpy.SpatialReference("NAD 1983 UTM Zone 20N"),
    transform_method        = geog_transform,
    in_coor_system          = arcpy.SpatialReference(4326),
    preserve_shape          = "NO_PRESERVE_SHAPE"
)
print(f"Projected: {pts_nad83}")

In [ ]:
# ── 4. REMOVE ALL ROWS EXCEPT STARTING POINT ───────────────────────────────────

# find the earliest start_datetime_UTC value in the projected points
with arcpy.da.SearchCursor(pts_nad83, ["start_datetime_UTC"]) as cur:
    earliest = min(row[0] for row in cur)

print(f"Earliest start_datetime_UTC: {earliest}")

# delete every row that is not the earliest one
with arcpy.da.UpdateCursor(pts_nad83, ["start_datetime_UTC"]) as cur:
    kept = False
    for row in cur:
        if row[0] == earliest and not kept:
            kept = True
        else:
            cur.deleteRow()

# confirm only one row remains
print(f"Rows remaining: {arcpy.management.GetCount(pts_nad83)[0]}")

In [ ]:
# ── 5. ADD LABEL FIELD ─────────────────────────────────────────────────────────

# add 'label' column (data type: text)
arcpy.management.AddField(
    in_table   = pts_nad83,
    field_name = "label",
    field_type = "TEXT"
)

# populate 'label' column with "Deployment Location"
with arcpy.da.UpdateCursor(pts_nad83, ["label"]) as cur:
    for row in cur:
        row[0] = "Deployment Location"
        cur.updateRow(row)

print("Label field added and populated.")